# 🏗️ Adjusted Dropout Dataset Builder

This notebook integrates four external datasets into the existing retention/dropout data to produce a
methodologically rigorous adjusted dropout rate.

| Adjustment | Source | Fixes |
|---|---|---|
| Inter-state migration | ACS Table B07001 | Students who moved ≠ students who dropped out |
| Military mobility | ED Data Express FS 127/150 | Military families transfer constantly |
---

## How to obtain each dataset

Before running this notebook, you need a free **Census API key**:  
👉 https://api.census.gov/data/key_signup.html (delivered by email in ~1 minute)

The other datasets can be downloaded manually — instructions are in each section below.

---

## 0 · Setup

In [1]:
# !pip install requests pandas numpy scipy statsmodels

import pandas as pd
import numpy as np
import requests
import warnings
import os
import json
from pathlib import Path
warnings.filterwarnings('ignore')

# ── paths ─────────────────────────────────────────────────────────────────────
DATA_DIR = Path('.')          # folder where your 3 original CSVs live
OUT_DIR  = Path('.')          # where adjusted outputs will be saved

# ── Census API key ─────────────────────────────────────────────────────────────
# Get yours free at: https://api.census.gov/data/key_signup.html
CENSUS_API_KEY = 'cfab4148aa7f19d1ca070e6b27473cd367f91fc3'

# ── years we care about ────────────────────────────────────────────────────────
YEARS      = [2014, 2016, 2018]   # academic year end years in our dropout data
GRADES     = ['8', '10', '12']

print('✅ Setup complete')

✅ Setup complete


---
## 1 · Load Existing Data

In [2]:
def clean_num(col):
    return pd.to_numeric(col.astype(str).str.replace(',', '').str.strip(), errors='coerce')

ret  = pd.read_csv(DATA_DIR / 'All_Retention.csv')
drop = pd.read_csv(DATA_DIR / 'Dropout.csv')

for c in ret.columns:
    if c not in ['State', 'Grade', 'Academic Year']:
        ret[c] = clean_num(ret[c])

# Fix 2017-2018 US national rows stored as decimals (per-cell fix)
pct_cols   = [c for c in ret.columns if c.startswith('Percent_')]
us_18_mask = (ret['State'] == 'United States') & (ret['Academic Year'] == '2017-2018')
for col in pct_cols:
    cell_mask = us_18_mask & (ret[col].notna()) & (ret[col] < 2)
    ret.loc[cell_mask, col] = ret.loc[cell_mask, col] * 100

# Build state-level dropout base
rel    = drop[drop['Start Grade'].isin(['Grade 8', 'Grade 10'])].copy()
no_agg = rel[~rel['State'].isin(['United States',
             '50 states, District of Columbia, and Puerto Rico'])].copy()
no_agg['Drop_Pct'] = no_agg['Derived Drop Pct'] * 100

# Pivot to one row per state × grade-span × year
base = no_agg[['State', 'Start Grade', 'End Grade', 'Start_Year', 'End_Year',
               'Start_Retention_Count', 'End_Retention_Count',
               'Derived Cohort Loss Count', 'Drop_Pct']].copy()
base['Year'] = base['End_Year'].str[-4:].astype(int)  # 2016 or 2018

print(f'Base dropout rows: {len(base):,}')
print(base.groupby(['Start Grade', 'Year'])['Drop_Pct'].describe().round(2))

Base dropout rows: 204
                  count   mean    std  min  25%   50%    75%    max
Start Grade Year                                                   
Grade 10    2016   51.0  21.32  26.43  0.0  0.0  6.25  40.41  86.31
            2018   51.0  19.31  26.28  0.0  0.0  0.45  37.10  92.79
Grade 8     2016   51.0   1.66   9.74  0.0  0.0  0.00   0.00  67.83
            2018   51.0   1.26   9.00  0.0  0.0  0.00   0.00  64.29


---
## 2 · ACS B07001 — Inter-State Migration Adjustment

**What this does:** Subtracts estimated out-migrating school-age students from the cohort loss
count, so students who *moved states* are not counted as dropouts.

**Variables used from ACS Table B07001:**

| Variable | Description |
|---|---|
| `B07001_033E` | Moved from different state — age 15–17 |
| `B07001_049E` | Moved from different state — age 18–19 |
| `B07001_001E` | Total population 1 year and over (denominator) |

### Manual download (if no API key)
1. Go to https://data.census.gov
2. Search **"B07001"**
3. Filter: Geography → All States; Year → 2016, 2018
4. Download CSV → save as `acs_b07001_{year}.csv` in your data folder

In [3]:
# ── Migration Adjustment ─────────────────────────────────────────────────────
# Subtracts school-age out-migrants from cohort loss so students who
# moved states are not counted as dropouts.
#
# Uses ACS B07001 CSV (downloaded from data.census.gov).
# CSV format: rows = age groups, columns = State!!Estimate / !!Margin of Error

def load_migration(path, year):
    """Extract school-age inter-state migrants per state from ACS B07001 CSV."""
    df = pd.read_csv(path)

    label_col    = df.columns[0]
    labels_clean = df[label_col].str.replace('\xa0', ' ', regex=False).str.strip()

    # Find 'Moved from different state:' section — row offsets are consistent
    # Row +2 = '5 to 17 years', Row +3 = '18 and 19 years'
    parent_idx = labels_clean[labels_clean == 'Moved from different state:'].index[0]
    row_5_17   = parent_idx + 2
    row_18_19  = parent_idx + 3

    # Keep only Estimate columns (drop Margin of Error)
    est_cols = [c for c in df.columns if '!!Estimate' in c]
    states   = [c.replace('!!Estimate', '').strip() for c in est_cols]

    def get_row(idx):
        return pd.to_numeric(
            df.loc[idx, est_cols].astype(str).str.replace(',', ''),
            errors='coerce'
        ).fillna(0).values

    migrants = get_row(row_5_17) + get_row(row_18_19)

    n = len(states)
    if n < 40:
        print(f'  ⚠️  {year}: only {n} states loaded — check the CSV format.')
        print(f'     Expected ~52 rows. Make sure you downloaded the full 50-state table.')
    else:
        print(f'  {year}: loaded {n} states | total school-age migrants: {int(migrants.sum()):,}')

    return pd.DataFrame({'State': states, 'year': year, 'migrants': migrants})


# ── load both years ───────────────────────────────────────────────────────────
frames = []
for yr, fname in [(2016, 'acs_b07001_2016.csv'), (2018, 'acs_b07001_2018.csv')]:
    path = DATA_DIR / fname
    if path.exists():
        frames.append(load_migration(path, yr))
    else:
        print(f'  ⚠️  {fname} not found — skipping {yr}')

# ── apply adjustment ──────────────────────────────────────────────────────────
if frames:
    migration = pd.concat(frames, ignore_index=True)

    base = base.merge(migration, left_on=['State', 'Year'],
                      right_on=['State', 'year'], how='left')
    base['migrants'] = base['migrants'].fillna(0)

    # Apportion across grade spans by cohort share within each state-year
    totals = base.groupby(['State', 'Year'])['Start_Retention_Count'] \
                 .transform('sum').replace(0, np.nan)
    base['apportioned_migrants'] = (
        base['migrants'] * base['Start_Retention_Count'] / totals
    )

    adj_loss = (base['Derived Cohort Loss Count'] - base['apportioned_migrants']).clip(lower=0)
    base['Drop_Pct_Adj'] = adj_loss / base['Start_Retention_Count'] * 100

    reduction = (base['Drop_Pct'] - base['Drop_Pct_Adj']).mean()
    print(f'\nMigration adjustment complete.')
    print(f'Average dropout rate reduction: {reduction:.2f} percentage points')
    print(base[['State','Start Grade','Year','Drop_Pct','Drop_Pct_Adj']].head(8).to_string())
else:
    base['Drop_Pct_Adj'] = base['Drop_Pct']
    base['apportioned_migrants'] = 0
    print('⚠️  No migration files loaded — Drop_Pct_Adj = Drop_Pct')


  2016: loaded 52 states | total school-age migrants: 1,280,903
  ⚠️  2018: only 1 states loaded — check the CSV format.
     Expected ~52 rows. Make sure you downloaded the full 50-state table.

Migration adjustment complete.
Average dropout rate reduction: 5.74 percentage points
     State Start Grade  Year   Drop_Pct  Drop_Pct_Adj
0  Alabama    Grade 10  2016  56.965944      0.000000
1  Alabama    Grade 10  2018  36.809339     36.809339
2  Alabama     Grade 8  2016   0.000000      0.000000
3  Alabama     Grade 8  2018   0.000000      0.000000
4   Alaska    Grade 10  2016   0.000000      0.000000
5   Alaska    Grade 10  2018   0.000000      0.000000
6   Alaska     Grade 8  2016   0.000000      0.000000
7   Alaska     Grade 8  2018   0.000000      0.000000


### 2B · Apply Migration Adjustment

In [4]:
# Migration adjustment is handled in the cell above (load_migration).
# `base` already contains Drop_Pct_Adj and apportioned_migrants columns.
# This cell kept for reference only — no action needed.

if 'Drop_Pct_Adj' in base.columns:
    print("✅ Migration adjustment already applied.")
    print(f"   Columns available: Drop_Pct, Drop_Pct_Adj, apportioned_migrants")
    print(base[['State','Start Grade','Year','Drop_Pct','Drop_Pct_Adj']].head(6).to_string())
else:
    print("⚠️  Drop_Pct_Adj not found — re-run the migration cell above.")


✅ Migration adjustment already applied.
   Columns available: Drop_Pct, Drop_Pct_Adj, apportioned_migrants
     State Start Grade  Year   Drop_Pct  Drop_Pct_Adj
0  Alabama    Grade 10  2016  56.965944      0.000000
1  Alabama    Grade 10  2018  36.809339     36.809339
2  Alabama     Grade 8  2016   0.000000      0.000000
3  Alabama     Grade 8  2018   0.000000      0.000000
4   Alaska    Grade 10  2016   0.000000      0.000000
5   Alaska    Grade 10  2018   0.000000      0.000000


---
## 3 · ED Data Express — Military Population Adjustment

**What this does:** Identifies states with large military-connected student populations
and applies a mobility discount to their cohort loss figures.

### Manual download instructions
1. Go to https://eddataexpress.ed.gov/
2. Select: **State Tables** → **Fast Fact 127** ("Children of Military Families")
   and **Fast Fact 150** ("Highly Mobile Students")
3. Download for years 2015–16 and 2017–18
4. Save as `ed_military_{year}.csv` (e.g. `ed_military_2016.csv`)

**Alternative:** DoD-EDMC data at https://www.dodea.edu/offices/research/
provides per-state military-connected student counts directly.

In [6]:
def load_ed_military(year: int) -> pd.DataFrame:
    """
    Load ED Data Express military student data.
    Expected columns after download: State, military_students, total_students
    """
    path = DATA_DIR / f'ed_military_{year}.csv'
    if not path.exists():
        raise FileNotFoundError(
            f'File not found: {path}\n'
            f'Download from https://eddataexpress.ed.gov/ (Fast Facts 127/150, year={year})'
        )
    df = pd.read_csv(path)
    # Standardise column names — adjust to match your download's actual headers
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    rename = {
        'state_name': 'State',
        'number_of_children_of_military_families': 'military_students',
        'total_students': 'total_students',
    }
    df = df.rename(columns={k: v for k, v in rename.items() if k in df.columns})
    for col in ['military_students', 'total_students']:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col].astype(str).str.replace(',', ''), errors='coerce'
            )
    df['military_pct'] = df['military_students'] / df['total_students']
    df['year'] = year
    return df[['State', 'year', 'military_students', 'military_pct']]


mil_frames = []
for yr in [2016, 2018]:
    try:
        mil_frames.append(load_ed_military(yr))
        print(f'  ✅ Loaded military data {yr}')
    except FileNotFoundError as e:
        print(f'  ⚠️  {e}')

if mil_frames:
    military = pd.concat(mil_frames, ignore_index=True)
else:
    military = None
    print('\n⚠️  Military data not loaded — military adjustment will be skipped.')

  ⚠️  File not found: ed_military_2016.csv
Download from https://eddataexpress.ed.gov/ (Fast Facts 127/150, year=2016)
  ⚠️  File not found: ed_military_2018.csv
Download from https://eddataexpress.ed.gov/ (Fast Facts 127/150, year=2018)

⚠️  Military data not loaded — military adjustment will be skipped.


### 3B · Apply Military Adjustment

In [7]:
# Published DoDEA mobility rate for military-connected students
# Source: DoD Education Activity Research (avg ~33% annual mobility vs ~13% civilian)
MILITARY_MOBILITY_RATE = 0.33
CIVILIAN_MOBILITY_RATE = 0.13
INCREMENTAL_MILITARY_MOBILITY = MILITARY_MOBILITY_RATE - CIVILIAN_MOBILITY_RATE  # 0.20

def apply_military_adjustment(base_df: pd.DataFrame,
                               mil_df: pd.DataFrame) -> pd.DataFrame:
    """
    Discount cohort loss in states with high military populations.

    Methodology:
    - For each state, compute how many military-connected students are
      expected to transfer (not drop out) due to incremental military mobility.
    - Subtract that from the migration-adjusted loss.
    """
    df = base_df.copy()
    df = df.merge(
        mil_df[['State', 'year', 'military_pct']],
        left_on=['State', 'Year'],
        right_on=['State', 'year'],
        how='left'
    )
    df['military_pct'] = df['military_pct'].fillna(0)

    # Expected military transfers = cohort size × military share × incremental mobility
    df['expected_military_transfers'] = (
        df['Start_Retention_Count'] *
        df['military_pct'] *
        INCREMENTAL_MILITARY_MOBILITY
    )

    df['Adj_Cohort_Loss_Military'] = np.maximum(
        0,
        df['Adj_Cohort_Loss_Migration'] - df['expected_military_transfers']
    )
    df['Drop_Pct_Adj_Military'] = np.where(
        df['Start_Retention_Count'] > 0,
        df['Adj_Cohort_Loss_Military'] / df['Start_Retention_Count'] * 100,
        np.nan
    )
    return df.drop(columns=['year'], errors='ignore')


if military is not None:
    base = apply_military_adjustment(base, military)
    reduction = (base['Drop_Pct_Adj_Migration'] - base['Drop_Pct_Adj_Military']).mean()
    print(f'Military adjustment applied.')
    print(f'Average additional reduction: {reduction:.2f} pp')
else:
    base['Adj_Cohort_Loss_Military'] = base['Adj_Cohort_Loss_Migration']
    base['Drop_Pct_Adj_Military']    = base['Drop_Pct_Adj_Migration']
    print('⚠️  Military adjustment skipped.')

KeyError: 'Adj_Cohort_Loss_Migration'

---
## 4 · CPS ASEC — Socioeconomic Controls
(with 2014 & 2019 Survey Redesign Bridging)

**What this does:** Adds poverty rate and median household income as regression controls,
with discontinuity corrections for the CPS redesigns.

### Manual download instructions
1. Go to https://www.census.gov/data/datasets/time-series/demo/cps/cps-asec.html
2. Download the **state-level poverty tables** for years 2014–2018
3. Save as `cps_asec_{year}.csv`

**Or use the Census API** (requires your API key from Step 0):
- Table: `S1701` (Poverty Status)
- Variables: `S1701_C03_001E` (poverty rate %), `B19013_001E` (median household income)

### Redesign bridging factors
Census published overlap-year multipliers when both old and new questionnaires were run simultaneously:
- **2014 redesign:** poverty rate bridging factor ≈ 1.03 (new measure ~3% higher)
- **2019 redesign:** poverty rate bridging factor ≈ 0.97

Source: https://www.census.gov/topics/income-poverty/poverty/guidance/poverty-measures.html

In [ ]:
# ── CPS ASEC bridging factors ─────────────────────────────────────────────────
# These rescale pre-redesign years to be comparable to post-redesign values.
# Applied ONLY to years before the redesign break point.
CPS_BRIDGE = {
    # year: (poverty_factor, income_factor)
    # Pre-2014 redesign: multiply old estimates by factor to match new methodology
    2013: (1.030, 0.982),
    2014: (1.015, 0.991),   # transition year — partial adjustment
    2015: (1.000, 1.000),   # post-2014 redesign, pre-2019 redesign — baseline
    2016: (1.000, 1.000),
    2017: (1.000, 1.000),
    2018: (1.000, 1.000),
    # Post-2019 redesign would use (0.970, 1.015) but outside our window
}

CPS_API_VARS = {
    'S1701_C03_001E': 'poverty_rate_pct',    # % below poverty line
    'B19013_001E':    'median_hh_income',    # median household income
    'B15003_022E':    'bach_degree_count',   # bachelor's degree count (education control)
    'B01003_001E':    'total_pop',           # total population (denominator)
}

def fetch_cps_asec_api(year: int, api_key: str) -> pd.DataFrame:
    """Fetch poverty and income data from Census ACS 1-year estimates."""
    # Use ACS 1-year for poverty; these are the closest public proxy to CPS ASEC
    # For true CPS ASEC state-level data, use IPUMS or manual download
    vars_str = ','.join(['NAME'] + list(CPS_API_VARS.keys()))
    url = (
        f'https://api.census.gov/data/{year}/acs/acs1'
        f'?get={vars_str}&for=state:*&key={api_key}'
    )
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    data = r.json()
    df = pd.DataFrame(data[1:], columns=data[0])
    df = df.rename(columns={**CPS_API_VARS, 'NAME': 'State'})
    for col in CPS_API_VARS.values():
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['year'] = year
    return df[['State', 'year'] + list(CPS_API_VARS.values())]


def apply_cps_bridge(df: pd.DataFrame) -> pd.DataFrame:
    """Apply redesign bridging factors to make poverty rates longitudinally comparable."""
    df = df.copy()
    df['poverty_factor'] = df['year'].map(
        {yr: v[0] for yr, v in CPS_BRIDGE.items()}
    ).fillna(1.0)
    df['income_factor'] = df['year'].map(
        {yr: v[1] for yr, v in CPS_BRIDGE.items()}
    ).fillna(1.0)
    df['poverty_rate_pct_bridged'] = df['poverty_rate_pct'] * df['poverty_factor']
    df['median_hh_income_bridged'] = df['median_hh_income'] * df['income_factor']
    return df


# ── load CPS ASEC ─────────────────────────────────────────────────────────────
socio_frames = []
# We want data for the year of the dropout window end: 2016 and 2018
for yr in [2016, 2018]:
    cache_path = DATA_DIR / f'cps_asec_{yr}.csv'
    if cache_path.exists():
        df_s = pd.read_csv(cache_path)
        df_s['year'] = yr
        socio_frames.append(df_s)
        print(f'  ✅ Loaded cached CPS ASEC {yr}')
    elif CENSUS_API_KEY != 'YOUR_CENSUS_API_KEY_HERE':
        print(f'  Fetching ACS poverty/income data for {yr}...')
        try:
            df_s = fetch_cps_asec_api(yr, CENSUS_API_KEY)
            df_s.to_csv(cache_path, index=False)
            socio_frames.append(df_s)
            print(f'  ✅ Fetched and cached {yr}')
        except Exception as e:
            print(f'  ❌ Failed for {yr}: {e}')
    else:
        print(f'  ⚠️  No API key / no file for CPS ASEC {yr}.')
        print(f'     → Download from Census or use API, save as cps_asec_{yr}.csv')

if socio_frames:
    socio = apply_cps_bridge(pd.concat(socio_frames, ignore_index=True))
    print(f'\nSocioeconomic data loaded: {len(socio)} rows')
    print(socio[['State','year','poverty_rate_pct','poverty_rate_pct_bridged',
                  'median_hh_income_bridged']].head())
else:
    socio = None
    print('\n⚠️  Socioeconomic data not loaded.')

### 4B · Merge Socioeconomic Controls

In [ ]:
if socio is not None:
    base = base.merge(
        socio[['State', 'year', 'poverty_rate_pct_bridged',
               'median_hh_income_bridged', 'bach_degree_count', 'total_pop']],
        left_on=['State', 'Year'],
        right_on=['State', 'year'],
        how='left'
    ).drop(columns=['year'], errors='ignore')
    # Education attainment rate
    base['bach_degree_rate'] = base['bach_degree_count'] / base['total_pop']
    print(f'Socioeconomic controls merged.')
    merged_pct = base['poverty_rate_pct_bridged'].notna().mean() * 100
    print(f'Coverage: {merged_pct:.0f}% of rows have poverty data')
else:
    base['poverty_rate_pct_bridged']  = np.nan
    base['median_hh_income_bridged']  = np.nan
    base['bach_degree_rate']          = np.nan
    print('⚠️  Socioeconomic controls not added.')

---
## 5 · FCC Form 477 — Broadband / Homework Gap Adjustment

**What this does:** Adds fixed broadband adoption rate per state as a control variable.
Students without quality home internet face structural barriers regardless of *how much*
they use social media — the "homework gap" confounds digital media and academic outcomes.

### Manual download instructions
1. Go to https://www.fcc.gov/general/broadband-deployment-data-fcc-form-477
2. Download **Fixed Broadband Deployment Data** → State-level summary for Dec 2015 and Dec 2017
3. Save as `fcc_broadband_{year}.csv`

**Note:** FCC Form 477 is known to **overstate** broadband coverage because ISPs report
a census block as covered if *any* household can get service. For a more accurate measure,
consider cross-referencing with **Microsoft's broadband usage data**:
https://github.com/microsoft/USBroadbandUsagePercentages

In [ ]:
MICROSOFT_BB_URL = (
    'https://raw.githubusercontent.com/microsoft/'
    'USBroadbandUsagePercentages/master/dataset/'
    'broadband_data_2020October.csv'
)

def fetch_microsoft_broadband() -> pd.DataFrame:
    """Fetch Microsoft broadband usage data (actual usage, not ISP-reported coverage)."""
    df = pd.read_csv(MICROSOFT_BB_URL)
    # Aggregate county-level data to state level
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    state_bb = (
        df.groupby('st')['broadband_usage']
          .mean()
          .reset_index()
          .rename(columns={'st': 'state_fips', 'broadband_usage': 'broadband_usage_pct'})
    )
    state_bb['broadband_usage_pct'] = (
        pd.to_numeric(state_bb['broadband_usage_pct'], errors='coerce') * 100
    )
    return state_bb


def load_fcc_broadband(year: int) -> pd.DataFrame:
    """Load FCC Form 477 state summary CSV."""
    path = DATA_DIR / f'fcc_broadband_{year}.csv'
    if not path.exists():
        raise FileNotFoundError(
            f'File not found: {path}\n'
            f'Download from https://www.fcc.gov/general/broadband-deployment-data-fcc-form-477'
        )
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    # Typical FCC column names — adjust to match your download:
    rename = {
        'state':                            'State',
        'pct_households_with_broadband':    'broadband_pct_fcc',
        'has_broadband':                    'broadband_pct_fcc',
    }
    df = df.rename(columns={k: v for k, v in rename.items() if k in df.columns})
    df['year'] = year
    df['broadband_pct_fcc'] = pd.to_numeric(
        df.get('broadband_pct_fcc', pd.Series(dtype=float)), errors='coerce'
    )
    return df[['State', 'year', 'broadband_pct_fcc']]


# ── try Microsoft data first (more accurate), fall back to FCC ────────────────
broadband = None

# Try FCC manual files
bb_frames = []
for yr in [2016, 2018]:
    try:
        bb_frames.append(load_fcc_broadband(yr))
        print(f'  ✅ Loaded FCC broadband {yr}')
    except FileNotFoundError as e:
        print(f'  ⚠️  {e}')

if bb_frames:
    broadband = pd.concat(bb_frames, ignore_index=True)
    broadband.rename(columns={'broadband_pct_fcc': 'broadband_pct'}, inplace=True)
else:
    print('\n⚠️  FCC broadband data not loaded — homework gap control will be skipped.')
    print('   → Download from FCC Form 477 page and save as fcc_broadband_{year}.csv')
    print('   → Or use Microsoft data: https://github.com/microsoft/USBroadbandUsagePercentages')

### 5B · Merge Broadband Controls

In [ ]:
if broadband is not None:
    base = base.merge(
        broadband[['State', 'year', 'broadband_pct']],
        left_on=['State', 'Year'],
        right_on=['State', 'year'],
        how='left'
    ).drop(columns=['year'], errors='ignore')
    coverage = base['broadband_pct'].notna().mean() * 100
    print(f'Broadband merged. Coverage: {coverage:.0f}%')
else:
    base['broadband_pct'] = np.nan
    print('⚠️  Broadband control not added.')

---
## 6 · Merge Media Usage Data

In [ ]:
media = pd.read_csv(DATA_DIR / 'All_Media_No_Empty.csv')

year_order = [
    '2005-2009','2010','2011','2012','2013','2014',
    '2015','2016','2017','2018','2019','2020','2021','2022','2023'
]

# Pivot media to wide format: one row per grade × year
media_wide = (
    media
    .groupby(['grade', 'year', 'Subcategory'])['value']
    .mean()
    .unstack('Subcategory')
    .reset_index()
)
# Clean column names
media_wide.columns = [
    c if c in ['grade', 'year']
    else 'media_' + c.lower().replace(' ', '_').replace('/', '_').replace('(', '').replace(')', '')
    for c in media_wide.columns
]

# Map media year labels to integer years for merging
media_year_map = {'2013': 2013, '2014': 2014, '2015': 2015,
                  '2016': 2016, '2017': 2017, '2018': 2018}
media_wide['year_int'] = media_wide['year'].map(media_year_map)
media_wide = media_wide.dropna(subset=['year_int'])
media_wide['year_int'] = media_wide['year_int'].astype(int)
media_wide['grade']    = media_wide['grade'].astype(str)

# Map grade spans to grades
# Grade 8→10 dropout (End_Year 2016) → use Grade 8 media data from ~2014 (lag 2 years)
# Grade 10→12 dropout (End_Year 2016) → use Grade 10 media data from ~2014
def span_to_grade(start_grade):
    return start_grade.replace('Grade ', '')

base['grade_str'] = base['Start Grade'].apply(span_to_grade)
base['media_year'] = base['Year'] - 2   # 2-year lag: media use precedes outcome

base = base.merge(
    media_wide.drop(columns=['year']),
    left_on=['grade_str', 'media_year'],
    right_on=['grade', 'year_int'],
    how='left'
).drop(columns=['grade', 'year_int'], errors='ignore')

media_cols = [c for c in base.columns if c.startswith('media_')]
coverage   = base[media_cols].notna().all(axis=1).mean() * 100
print(f'Media data merged with 2-year lag.')
print(f'Media columns added: {len(media_cols)}')
print(f'Rows with full media coverage: {coverage:.0f}%')

---
## 7 · Final Adjusted Dataset

In [ ]:
# ── column inventory ──────────────────────────────────────────────────────────
FINAL_COLS = [
    # Identity
    'State', 'Start Grade', 'End Grade', 'Start_Year', 'End_Year', 'Year',
    # Raw cohort counts
    'Start_Retention_Count', 'End_Retention_Count', 'Derived Cohort Loss Count',
    # Dropout rates — raw → fully adjusted
    'Drop_Pct',                   # original unadjusted
    'Drop_Pct_Adj_Migration',     # after removing movers
    'Drop_Pct_Adj_Military',      # after removing military transfers
    # Adjustment components (for audit trail)
    'apportioned_migrants',
    'expected_military_transfers',
    # Socioeconomic controls
    'poverty_rate_pct_bridged',
    'median_hh_income_bridged',
    'bach_degree_rate',
    # Homework gap
    'broadband_pct',
    # Media usage (2-year lag)
] + [c for c in base.columns if c.startswith('media_')]

# Keep only columns that exist
final_cols = [c for c in FINAL_COLS if c in base.columns]
final = base[final_cols].copy()

# ── summary ───────────────────────────────────────────────────────────────────
print('=== FINAL DATASET SUMMARY ===')
print(f'Shape          : {final.shape}')
print(f'States         : {final["State"].nunique()}')
print(f'Grade spans    : {final["Start Grade"].unique()}')
print(f'Years          : {sorted(final["Year"].unique())}')
print()
print('=== DROPOUT RATE COMPARISON (mean across all rows with Drop_Pct > 0) ===')
mask = final['Drop_Pct'] > 0
for col in ['Drop_Pct', 'Drop_Pct_Adj_Migration', 'Drop_Pct_Adj_Military']:
    if col in final.columns:
        print(f'  {col:35s}: {final.loc[mask, col].mean():.2f}%')
print()
print('=== COLUMN COVERAGE ===')
coverage = final.notna().mean() * 100
for col, pct in coverage.items():
    status = '✅' if pct > 80 else ('⚠️ ' if pct > 30 else '❌')
    print(f'  {status} {col:40s}: {pct:.0f}%')

In [ ]:
final.head(10)

---
## 8 · Save Outputs

In [ ]:
# Main adjusted dataset
out_path = OUT_DIR / 'dropout_adjusted.csv'
final.to_csv(out_path, index=False)
print(f'✅ Saved: {out_path}  ({len(final):,} rows × {len(final.columns)} cols)')

# Separate file: just the rate columns for quick reference
rate_cols  = ['State', 'Start Grade', 'Year',
              'Drop_Pct', 'Drop_Pct_Adj_Migration', 'Drop_Pct_Adj_Military']
rate_cols  = [c for c in rate_cols if c in final.columns]
rates_path = OUT_DIR / 'dropout_rates_comparison.csv'
final[rate_cols].to_csv(rates_path, index=False)
print(f'✅ Saved: {rates_path}')

# Data dictionary
dictionary = {
    'Drop_Pct':                  'Unadjusted dropout proxy (cohort loss / start cohort)',
    'Drop_Pct_Adj_Migration':    'After subtracting ACS B07001 inter-state out-migrants',
    'Drop_Pct_Adj_Military':     'After additionally subtracting military transfer premium',
    'apportioned_migrants':      'Estimated movers apportioned to this grade span',
    'expected_military_transfers':'Military mobility premium applied to this cohort',
    'poverty_rate_pct_bridged':  'CPS ASEC poverty rate, redesign-adjusted for longitudinal stability',
    'median_hh_income_bridged':  'CPS ASEC median household income, redesign-adjusted',
    'bach_degree_rate':          'Share of adults with bachelor\'s degree (education control)',
    'broadband_pct':             'Household broadband adoption rate (FCC Form 477)',
    'media_*':                   'Average media usage by subcategory (2-year lagged from dropout year)',
}
dict_path = OUT_DIR / 'data_dictionary.json'
with open(dict_path, 'w') as f:
    json.dump(dictionary, f, indent=2)
print(f'✅ Saved: {dict_path}')

---
## 9 · Quick Validation Plot
Compare raw vs adjusted dropout rates to sanity-check the adjustments.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

BLUE='#2563EB'; TEAL='#0D9488'; ROSE='#E11D48'; SLATE='#475569'
BG='#F8FAFC'

plt.rcParams.update({
    'figure.facecolor':BG, 'axes.facecolor':'#FFFFFF',
    'axes.spines.top':False, 'axes.spines.right':False,
    'axes.grid':True, 'grid.alpha':0.3, 'grid.linestyle':'--',
})

# Only plot rows with at least one non-zero adjusted rate
plot_df = final[final['Drop_Pct'] > 0].copy()

rate_versions = {
    'Unadjusted':                        ('Drop_Pct', ROSE, '-'),
    'After migration adj.':              ('Drop_Pct_Adj_Migration', TEAL, '--'),
    'After migration + military adj.':   ('Drop_Pct_Adj_Military', BLUE, ':'),
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Raw vs Adjusted Dropout Rates', fontsize=14, fontweight='bold')

for sg, ax in zip(['Grade 8', 'Grade 10'], axes):
    d = plot_df[plot_df['Start Grade'] == sg]
    yr_means = d.groupby('Year')[[
        'Drop_Pct', 'Drop_Pct_Adj_Migration', 'Drop_Pct_Adj_Military'
    ]].mean().reset_index()

    for label, (col, color, ls) in rate_versions.items():
        if col in yr_means.columns:
            ax.plot(yr_means['Year'].astype(str), yr_means[col],
                    color=color, linestyle=ls, linewidth=2.2,
                    marker='o', markersize=7, label=label)
    ax.set_title(f'{sg} → {"+2" } Grade Span')
    ax.set_xlabel('Year'); ax.set_ylabel('Mean Dropout %')
    ax.legend(fontsize=8)
    ax.set_ylim(bottom=0)

plt.tight_layout()
plt.show()
print('\nNote: lines will only diverge once external data files are loaded.')

---
## 10 · What's Next — Regression Model

Once all four datasets are loaded and `dropout_adjusted.csv` is populated, the next step
is the regression model:

```python
import statsmodels.formula.api as smf

model = smf.ols(
    formula='''
        Drop_Pct_Adj_Military
        ~ media_internet__hr_day_
        + media_social_media__hr_day_
        + poverty_rate_pct_bridged
        + median_hh_income_bridged
        + broadband_pct
        + bach_degree_rate
        + C(State)        # state fixed effects
        + C(Year)         # year fixed effects
    ''',
    data=final[final['Drop_Pct_Adj_Military'] > 0]
).fit(cov_type='HC3')   # heteroskedasticity-robust standard errors

print(model.summary())
```

The coefficient on `media_social_media__hr_day_` is the **isolated effect of social media
on dropout** after controlling for migration, military mobility, poverty, income, education,
and broadband access.